# EASM scan: badssl.com

Triggers a scan through the discovery-scheduler pod, tails the pipeline pod logs, and queries the Postgres gold tables.
Requires `kubectl` pointed at the local kind cluster (namespace `easm`) and `pandas`.

> The deployments currently run with `OFFLINE_MODE=true` (synthetic CT records and probes). For real discovery/probing of wikimedia.org set `LIVE = True` in section 1.

In [1]:
import json, subprocess, time
from io import StringIO
import pandas as pd

NS = "easm"
TENANT = "tenant_badssl"
DOMAIN = "wikimedia.org"

def kubectl(*args, check=True):
    r = subprocess.run(["kubectl", "-n", NS, *args], capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout

def sql(query):
    out = kubectl("exec", "deploy/postgres", "--", "psql", "-U", "easm", "-d", "easm", "--csv", "-c", query)
    return pd.read_csv(StringIO(out))

def logs(app, since="10m", tail=200, grep=None):
    """Logs from all pods labelled app=<name>, prefixed with the pod name."""
    out = kubectl("logs", "-l", f"app={app}", "--prefix", f"--since={since}", f"--tail={tail}", "--max-log-requests=10", check=False)
    lines = [l for l in out.splitlines() if not grep or grep in l]
    print("\n".join(lines) or f"(no logs for app={app})")

print(kubectl("get", "pods"))

NAME                                      READY   STATUS    RESTARTS       AGE
asset-api-6d8c6b7897-5qggv                1/1     Running   0              103s
discovery-scheduler-56f5d74b8b-8klfd      1/1     Running   0              44s
postgres-79967df69f-bgktx                 1/1     Running   1 (137m ago)   138m
probe-worker-c8bbbbdc6-59727              1/1     Running   0              43s
probe-worker-c8bbbbdc6-pwcw7              1/1     Running   0              42s
probe-worker-c8bbbbdc6-rlr4d              1/1     Running   0              43s
redis-79cfc78c7-mbwcv                     1/1     Running   0              131m
redpanda-8f59ffbfb-5kw9j                  1/1     Running   1 (138m ago)   138m
spark-stream-processor-65f86f4986-wgzph   1/1     Running   0              104s



## 1. (Optional) Enable live mode
Switches scheduler and probe workers to `OFFLINE_MODE=false` (queries crt.sh and probes wikimedia.org hosts over DNS/TLS/HTTP). Patches the running deployments only; `the k8s manifests` is unchanged.

In [2]:
LIVE = True
if LIVE:
    for d in ("discovery-scheduler", "probe-worker"):
        kubectl("set", "env", f"deploy/{d}", "OFFLINE_MODE=false")
        kubectl("rollout", "status", f"deploy/{d}", "--timeout=180s")

## 2. Trigger the scan

In [3]:
payload = json.dumps({"tenant_id": TENANT, "root_domain": DOMAIN})
script = (
    "import urllib.request as u;"
    f"r=u.Request('http://localhost:8000/scans',data={payload!r}.encode(),headers={{'Content-Type':'application/json'}});"
    "print(u.urlopen(r,timeout=120).read().decode())"
)
# crt.sh is occasionally down (502); the scheduler returns 500 when its retries are exhausted.
for attempt in range(1, 6):
    try:
        print(kubectl("exec", "deploy/discovery-scheduler", "--", "python", "-c", script))
        break
    except RuntimeError as e:
        print(f"attempt {attempt} failed (crt.sh likely unavailable): {str(e).splitlines()[-2] if len(str(e).splitlines())>1 else e}")
        time.sleep(30)
else:
    print("Scan not triggered; continuing with assets from earlier scans (if any).")

attempt 1 failed (crt.sh likely unavailable): urllib.error.HTTPError: HTTP Error 500: Internal Server Error


attempt 2 failed (crt.sh likely unavailable): urllib.error.HTTPError: HTTP Error 500: Internal Server Error


attempt 3 failed (crt.sh likely unavailable): urllib.error.HTTPError: HTTP Error 500: Internal Server Error


attempt 4 failed (crt.sh likely unavailable): urllib.error.HTTPError: HTTP Error 500: Internal Server Error


attempt 5 failed (crt.sh likely unavailable): urllib.error.HTTPError: HTTP Error 500: Internal Server Error


Scan not triggered; continuing with assets from earlier scans (if any).


## 3. Pod logs

In [4]:
print("=== discovery-scheduler ==="); logs("discovery-scheduler", since="5m")
print("\n=== probe-worker (x3) ==="); logs("probe-worker", since="5m")

=== discovery-scheduler ===
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]     with map_httpcore_exceptions():
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]          ^^^^^^^^^^^^^^^^^^^^^^^^^
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]   File "/usr/local/lib/python3.12/contextlib.py", line 158, in __exit__
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]     self.gen.throw(value)
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]   File "/usr/local/lib/python3.12/site-packages/httpx/_transports/default.py", line 118, in map_httpcore_exceptions
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler]     raise mapped_exc(message) from exc
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler] httpx.ReadTimeout
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler] 
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler] The above exception was the direct cause of the following exception:
[pod/discovery-scheduler-56f5d74b8b-8klfd/scheduler] 
[pod/discovery-scheduler-56f5

In [5]:
print("=== spark-stream-processor ==="); logs("spark-stream-processor", since="5m", tail=100)
print("\n=== asset-api ==="); logs("asset-api", since="5m", tail=50)

=== spark-stream-processor ===
[pod/spark-stream-processor-65f86f4986-wgzph/spark-driver] 26/11/28 09:45:45 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 40362 milliseconds

=== asset-api ===


(no logs for app=asset-api)


## 4. Wait for results in Postgres
Spark micro-batches sync to Postgres, so this can take a minute or two.

In [6]:
assets = pd.DataFrame()
for _ in range(30):
    assets = sql(f"SELECT * FROM gold_assets WHERE tenant_id = '{TENANT}' ORDER BY fqdn")
    if len(assets):
        break
    time.sleep(10)
print(f"{len(assets)} assets")

0 assets


## 5. Query tables

In [7]:
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 80)
assets

,asset_id,tenant_id,fqdn,cluster_id,root_cause_node,latest_ips_json,is_shared_cdn,detected_tech_json,tls_version,cert_sha256,status,exposure_type,validation_tier,first_seen,last_seen


In [8]:
sql(f"SELECT * FROM gold_asset_findings WHERE tenant_id = '{TENANT}'")

,asset_id,tenant_id,source,exposure_type,validation_tier,detected_at


In [9]:
sql(f"SELECT * FROM gold_asset_edges WHERE tenant_id = '{TENANT}'")

,edge_id,tenant_id,src_node,relation,dst_node,is_shared_infra,observed_at


In [10]:
# Row counts for all tables, including staging
tables = sql("SELECT tablename FROM pg_tables WHERE schemaname='public' ORDER BY 1")["tablename"]
pd.Series({t: int(sql(f"SELECT count(*) AS n FROM {t}")["n"][0]) for t in tables}, name="rows")

gold_asset_clusters_staging      9
gold_asset_edges               337
gold_asset_edges_staging        33
gold_asset_findings              5
gold_asset_findings_staging      0
gold_asset_sources_staging       4
gold_assets                    172
gold_assets_staging              3
Name: rows, dtype: int64

## 6. Serving API (clusters)

In [11]:
script = f"import urllib.request as u;print(u.urlopen('http://localhost:8080/tenants/{TENANT}/clusters',timeout=30).read().decode())"
print(json.dumps(json.loads(kubectl("exec", "deploy/asset-api", "--", "python", "-c", script)), indent=2))

[]
